# LLM2-Lab — campanhas Kaggle T4

Duas T4 são GPUs independentes, sem VRAM unificada. O protocolo de referência usa FP32.
Este notebook apresenta controles da campanha 0015; não representa uma nova hipótese aprovada.
A execução fica desligada por padrão. Consulte `docs/KAGGLE_T4_CAMPAIGN.md` e fixe um commit
publicado da branch experimental. O ZIP de análise não contém checkpoints retomáveis.


In [ ]:
import json

RUN_CAMPAIGN = False
# Revision of experiment/fractal-lrm for Cycle 1
LLM2_REVISION = "7b773cdd30bde04b62164796eec644af72909161"
LLM2_SETTINGS = {
    "mode": "treinar",
    "models": [
        "baseline_small",
        "lrm_no_jepa_k2",
        "lrm_no_jepa_k2_rms",
        "flrm_compressed_native_cached_no_jepa_k2",
        "flrm_compressed_native_cached_no_jepa_k2_rms"
    ],
    "seeds": [71, 72, 73],
    "steps": 2000,
    "batch_size": 32,
    "grad_accum_steps": 1,
    "compile": True,
    "paired_common_init": True,
    "evaluate_all_checkpoints": True,
    "diagnose_init": True,
    "diagnose_steps": [500, 1000, 1500, 2000],
    "diagnose_batch_size": 4,
    "max_seconds": 1800,
    "cpu_tests": True,
    "cuda_tests": True,
}
print("Execution enabled:", RUN_CAMPAIGN)
print("Pinned source:", LLM2_REVISION)
print(json.dumps(LLM2_SETTINGS, indent=2))


In [ ]:
if not RUN_CAMPAIGN:
    print("Planning only. Set RUN_CAMPAIGN=True explicitly to run on Kaggle.")
else:
    import os
    import re
    from urllib.request import urlopen

    if not re.fullmatch(r"[0-9a-f]{40}", LLM2_REVISION):
        raise ValueError("Select a complete 40-character commit SHA.")
    os.environ["LLM2_REVISION"] = LLM2_REVISION
    url = ("https://raw.githubusercontent.com/PedroGGC/-LLM-2-/"
           f"{LLM2_REVISION}/scripts/kaggle_notebook.py")
    with urlopen(url, timeout=60) as response:
        bootstrap = response.read(1024 * 1024 + 1)
    if len(bootstrap) > 1024 * 1024:
        raise ValueError("Unexpected bootstrap size.")
    exec(compile(bootstrap.decode("utf-8"), url, "exec"), globals())


## Preservar resultados

O helper registra código/configuração, testes, avaliações e logs. Salve os outputs completos
para preservar pesos e otimizador. Baixe o ZIP de análise para `artifacts/campaigns/`.
Para avaliar resultados existentes, selecione `avaliar_salvos` e `previous_campaign`
com o diretório dos checkpoints completos; um ZIP de análise sozinho não basta.
